In [24]:
from multiprocessing import freeze_support

import os
import sys
import pickle
import pprint
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

sys.path.append("../")

import torch as t
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

from tqdm import tqdm
from datetime import datetime
import json
import wandb

from devinterp.slt import estimate_learning_coeff, estimate_learning_coeff_with_summary
from devinterp.optim.sgld import SGLD
from devinterp.slt import sample
from devinterp.slt.llc import OnlineLLCEstimator
from devinterp.slt.wbic import OnlineWBICEstimator

from approxngd import KFAC
from PyHessian.pyhessian import *
from PyHessian.density_plot import *
from general_utils import *
from hessian_utils import *
from architectures.NN import *
from data.build_data import build_data

import plotly.express as px
import plotly.graph_objects as go
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

device = "cuda" if t.cuda.is_available() else "cpu"
print(device)

cuda


In [10]:
%load_ext autoreload
%autoreload
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
# create list of models and specify optimiser

hidden_nodes = [2, 4, 8, 16, 32]
models = {}
for hidden_node in hidden_nodes:
    models[hidden_node] = LinearMNIST(hidden_nodes=hidden_node, hidden_layers=3).to(device)

In [19]:
hyperparams = {
    "batch_size" : 128,
    "num_workers" : 12,
    "num_epochs" : 10,
    "lr" : 1e-03,
}

In [5]:
# load data from pytorch dataloaders

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = t.utils.data.DataLoader(train_set, batch_size=hyperparams["batch_size"], shuffle=True, num_workers=hyperparams["num_workers"], persistent_workers=True)
test_loader = t.utils.data.DataLoader(test_set, batch_size=hyperparams["batch_size"], shuffle=False, num_workers=hyperparams["num_workers"], persistent_workers=True)

In [13]:
# specify loss function
criterion = nn.CrossEntropyLoss()

In [20]:
# training loop

train_losses = {}
test_losses = {}
for hidden_node, model in models.items():
    print(f"TRAINING WITH {hidden_node} HIDDEN NODES")
    train_losses[hidden_node] = []
    test_losses[hidden_node] = []
    optimiser = t.optim.Adam(model.parameters(), lr=hyperparams["lr"])
    for epoch in range(1, hyperparams["num_epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimiser, criterion, device)
        test_loss = evaluate(model, test_loader, criterion, device)
        train_losses[hidden_node].append(train_loss)
        test_losses[hidden_node].append(test_loss)
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}: train_loss={train_loss:.4f}, test_loss={test_loss:.4f}")

TRAINING WITH 2 HIDDEN NODES


100%|██████████| 469/469 [00:04<00:00, 113.01it/s]


Epoch 2/10: train_loss=1.7418, test_loss=1.3586


100%|██████████| 469/469 [00:03<00:00, 150.01it/s]


Epoch 3/10: train_loss=1.3032, test_loss=1.2084


...

Epoch 8/10: train_loss=0.2724, test_loss=0.2745


100%|██████████| 469/469 [00:03<00:00, 150.90it/s]


Epoch 9/10: train_loss=0.2689, test_loss=0.2807


100%|██████████| 469/469 [00:04<00:00, 98.54it/s] 


Epoch 10/10: train_loss=0.2663, test_loss=0.2844


In [22]:
# visualise training and testing data
epochs = np.arange(1, hyperparams["num_epochs"]+1)
train_fig = go.Figure()
for hidden_node, train_loss in train_losses.items():
    train_fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode='lines+markers', name=str(hidden_node)))

train_fig.update_layout(title="Training loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Hidden nodes"
                  )
train_fig.show()

test_fig = go.Figure()
for hidden_node, test_loss in test_losses.items():
    test_fig.add_trace(go.Scatter(x=epochs, y=test_loss, mode='lines+markers', name=str(hidden_node)))

test_fig.update_layout(title="Test loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Hidden nodes",
                  )
test_fig.show()

In [27]:
# perform LLC estimation at convergence

NUM_CHAINS = 5
NUM_DRAWS = 400

optim_kwargs = dict(
    lr=1e-05,
    noise_level=1.0,
    elasticity=10,
    num_samples=len(train_set),
    temperature="adaptive",
)

models_results = {}
for hidden_node, model in models.items():
    results = estimate_learning_coeff_with_summary(
        model=model,
        loader=train_loader,
        sampling_method=SGLD,
        criterion=criterion,
        optimizer_kwargs=optim_kwargs,
        num_chains=NUM_CHAINS,
        num_draws=NUM_DRAWS,
        device=device,
        online=True,
    )
    models_results[hidden_node] = models_results

Chain 4: 100%|██████████| 400/400 [00:04<00:00, 91.34it/s] 


In [ ]:
LLC_fig = go.Figure()
for hidden_node, results in models_results.items():
    LLC_fig.add_trace(go.Scatter(
        x=np.arange(1, NUM_DRAWS+1),
        y=results['llc/means'],
        mode='lines+markers',
        name=str(hidden_node),
    ))
    LLC_fig.update_layout(title="LLC estimation evolution",
                  xaxis_title="Draws",
                  yaxis_title="LLC",
                  legend_title="Hidden nodes",
                  )
